In [2]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null


E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/o/openjdk-17/openjdk-17-jre-headless_17.0.20%2b8-1%7e24.04_amd64.deb  404  Not Found [IP: 91.189.91.81 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/o/openjdk-17/openjdk-17-jdk-headless_17.0.20%2b8-1%7e24.04_amd64.deb  404  Not Found [IP: 91.189.91.81 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?


In [3]:
!pip install pyspark -q

In [4]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, when

spark = SparkSession.builder \
    .appName("DataLakeAssignment") \
    .getOrCreate()

print(spark.version)


4.0.4


In [5]:
from google.colab import files

uploaded = files.upload()

Saving sales.csv to sales.csv


In [6]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("sales.csv")

df.show(10)
df.printSchema()

print("Total rows:", df.count())
print("Total columns:", len(df.columns))

+--------+-----------+---------------+--------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|  customer_name|             product| category|quantity|unit_price|discount_percent|  payment_method|      city|         state|order_date|order_status|
+--------+-----------+---------------+--------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|  100001|      C0256| Keerthi Mishra|               Saree| Clothing|       2|      6710|              20|     Credit Card|    Mumbai|   Maharashtra|2026-01-01|   Delivered|
|  100002|      C0042|    Arjun Singh|             T-Shirt| Clothing|       6|       680|               5|      Debit Card|   Chennai|    Tamil Nadu|2025-08-03|   Delivered|
|  100003|      C0348|    Keerthi Rao|              Hoodie| Clothing|       6|      3080|              12|             UPI|      P

In [7]:
import shutil
import os

os.makedirs("bronze", exist_ok=True)

shutil.copy("sales.csv", "bronze/sales.csv")

'bronze/sales.csv'

In [8]:
from pyspark.sql.functions import col, trim, when

print("Schema:")
df.printSchema()

print("Missing values:")
df.select([
    __import__("pyspark").sql.functions.count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()

print("Duplicate rows:", df.count() - df.dropDuplicates().count())

Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

Missing values:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|

In [9]:
df.show(20, truncate=False)
df.describe().show()

+--------+-----------+----------------+----------------------+-----------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name   |product               |category   |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+----------------+----------------------+-----------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra  |Saree                 |Clothing   |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh     |T-Shirt               |Clothing   |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao     |Hoodie                |Clothing   |6       |3080      |12         

In [10]:
from pyspark.sql.functions import col, trim, when

silver_df = df

for c, t in silver_df.dtypes:
    if t == "string":
        silver_df = silver_df.withColumn(
            c,
            when(trim(col(c)) == "", None)
            .otherwise(trim(col(c)))
        )

silver_df = silver_df.dropDuplicates()

silver_df.show(10, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka  |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra|2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books          |6       |6

In [11]:
print("Original rows:", df.count())
print("Silver rows:", silver_df.count())

print("Silver schema:")
silver_df.printSchema()

print("Remaining null values:")
silver_df.select([
    __import__("pyspark").sql.functions.count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in silver_df.columns
]).show()

silver_df.show(10, truncate=False)

Original rows: 1000
Silver rows: 990
Silver schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

Remaining null values:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+-

In [12]:
import os
import glob
import shutil

silver_df.coalesce(1).write \
    .option("header", True) \
    .mode("overwrite") \
    .csv("silver_temp")

os.makedirs("silver", exist_ok=True)

part_file = glob.glob("silver_temp/part-*.csv")[0]
shutil.copy(part_file, "silver/sales_silver.csv")

print("Silver CSV saved successfully")

Silver CSV saved successfully


In [13]:
files.download("silver/sales_silver.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>